# Arabic Cursive OCR Benchmark — Findings

Rich analysis over the whole `results/` tree: coverage, error anatomy,
script & letter confusions, diacritics, difficulty, model landscape,
operations (cost / latency / failures) and corpus health. Every chart
is driven by the same `arabic_ocr_bench.analysis.findings.build()` used
by the HTML leaderboard, so the notebook and the site never disagree.
Works in Colab or locally — point `RESULTS` at a folder of run dirs.

In [ ]:
# Colab only: mount Drive to read saved runs. Skip if running locally.
try:
    from google.colab import drive; drive.mount('/content/drive')
    RESULTS = '/content/drive/MyDrive/arabic-ocr-bench/results'
except Exception:
    RESULTS = 'results'
print('results dir:', RESULTS)

In [ ]:
# seaborn just for nicer defaults — every plot below also works without it.
try:
    import seaborn as sns; sns.set_theme()
except Exception:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'seaborn'])
    try:
        import seaborn as sns; sns.set_theme()
    except Exception:
        print('seaborn unavailable — using bare matplotlib')
import matplotlib.pyplot as plt

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
from arabic_ocr_bench.analysis.findings import build
F = build(RESULTS)
print({k: type(v).__name__ for k, v in F.items()})

## 1. Coverage

In [ ]:
# scored / attempted per (model, dataset) — this grid gates every number below.
cov = F.get('operational', {}).get('coverage', {})
if not cov.get('grid'):
    print('coverage not available')
else:
    import numpy as np
    grid = cov['grid']; models = cov.get('models', []); datasets = cov.get('datasets', [])
    ratio = np.array([[ (c[0] / c[1] if c[1] else np.nan) for c in row] for row in grid], float)
    fig, ax = plt.subplots(figsize=(2 + 1.1 * len(datasets), 1.4 + 0.5 * len(models)))
    im = ax.imshow(ratio, vmin=0, vmax=1, cmap='viridis', aspect='auto')
    ax.set_xticks(range(len(datasets))); ax.set_xticklabels(datasets, rotation=45, ha='right')
    ax.set_yticks(range(len(models))); ax.set_yticklabels(models)
    for i in range(len(models)):
        for j in range(len(datasets)):
            s, a = grid[i][j]
            ax.text(j, i, f'{s}/{a}', ha='center', va='center', color='w', fontsize=8)
    fig.colorbar(im, ax=ax, label='scored / attempted')
    ax.set_title('Coverage'); plt.show()

## 2. Error anatomy

In [ ]:
ea = F.get('error_anatomy', {})
if not ea:
    print('error anatomy not available')
else:
    import numpy as np
    sid = ea.get('sid_split', {})
    ms = [m for m in sid if m != '__pooled__']
    if ms:
        sub = np.array([sid[m].get('sub', 0) for m in ms], float)
        ins = np.array([sid[m].get('ins', 0) for m in ms], float)
        dele = np.array([sid[m].get('del', 0) for m in ms], float)
        x = np.arange(len(ms))
        fig, ax = plt.subplots()
        ax.bar(x, sub, label='sub')
        ax.bar(x, ins, bottom=sub, label='ins')
        ax.bar(x, dele, bottom=sub + ins, label='del')
        ax.set_xticks(x); ax.set_xticklabels(ms); ax.legend()
        ax.set_title('Sub / ins / del per model'); plt.show()
    else:
        print('no sid split')
    lb = ea.get('length_bias', {})
    deltas = [p[0] - p[1] for m in lb.values() for p in m.get('pred_vs_gt', [])]
    if deltas:
        plt.figure(); plt.hist(deltas, bins=20)
        plt.xlabel('pred_len - gt_len (chars)'); plt.title('Length bias (all models)'); plt.show()
    else:
        print('no length-bias deltas')
    rw = ea.get('rewrites', {}).get('char', [])
    print('top char rewrites (ref -> hyp, count):')
    for row in rw[:15]:
        print(f'  {row[0]!r} -> {row[1]!r}   x{row[2]}')

## 3. Script & letters

In [ ]:
sl = F.get('script_letters', {})
if not sl:
    print('script/letters not available')
else:
    import numpy as np
    pooled = sl.get('confusion', {}).get('pooled', [])
    if pooled:
        chars = sorted({c for r in pooled for c in (r[0], r[1])})
        idx = {c: i for i, c in enumerate(chars)}
        M = np.zeros((len(chars), len(chars)))
        for ref, hyp, cnt in pooled:
            M[idx[ref], idx[hyp]] += cnt
        fig, ax = plt.subplots(figsize=(2 + 0.5 * len(chars), 2 + 0.5 * len(chars)))
        im = ax.imshow(M, cmap='magma')
        ax.set_xticks(range(len(chars))); ax.set_xticklabels(chars)
        ax.set_yticks(range(len(chars))); ax.set_yticklabels(chars)
        ax.set_xlabel('hyp'); ax.set_ylabel('ref'); ax.set_title('Confusion (pooled)')
        fig.colorbar(im, ax=ax); plt.show()
    else:
        print('no confusion pairs')
    ler = sl.get('letter_error_rate', {}).get('pooled', [])
    if ler:
        ch = [r[0] for r in ler]; rate = [r[3] for r in ler]
        plt.figure(figsize=(2 + 0.4 * len(ch), 3)); plt.bar(range(len(ch)), rate)
        plt.xticks(range(len(ch)), ch); plt.ylabel('error rate')
        plt.title('Letter error rate (pooled)'); plt.show()
    rg = sl.get('rasm_gap', {})
    pts = [(f'{m}/{ds}', d.get('cer_light', 0.0), d.get('cer_rasm', 0.0))
           for m, per in rg.items() for ds, d in per.items()]
    if pts:
        y = np.arange(len(pts))
        light = [p[1] for p in pts]; rasm = [p[2] for p in pts]
        plt.figure(figsize=(6, 1.5 + 0.4 * len(pts)))
        for i in y:
            plt.plot([rasm[i], light[i]], [i, i], color='gray', zorder=1)
        plt.scatter(light, y, label='cer_light', zorder=2)
        plt.scatter(rasm, y, label='cer_rasm', zorder=2)
        plt.yticks(y, [p[0] for p in pts]); plt.legend()
        plt.title('Rasm gap (light vs rasm-only CER)'); plt.show()

## 4. Diacritics

In [ ]:
di = F.get('diacritics', {})
pm = di.get('per_mark', {}) if di else {}
marks = sorted({mk for m in pm.values() for mk in m})
if not marks:
    print('diacritics per-mark not available')
else:
    import numpy as np
    cats = ['missed', 'inserted', 'substituted']
    vals = {c: [sum(pm[m].get(mk, {}).get(c, 0) for m in pm) for mk in marks] for c in cats}
    x = np.arange(len(marks)); w = 0.25
    plt.figure()
    for k, c in enumerate(cats):
        plt.bar(x + (k - 1) * w, vals[c], width=w, label=c)
    plt.xticks(x, marks); plt.legend()
    plt.title('Diacritic errors by mark (summed over models)'); plt.show()

## 5. Difficulty

In [ ]:
df = F.get('difficulty', {})
if not df:
    print('difficulty not available')
else:
    hard = df.get('consensus_hard', [])
    print('consensus hard (top 10):')
    for r in hard[:10]:
        print(f"  {r['sample_id']}  mean_cer={r['mean_cer']:.3f}  n={r['n_models']}  gt={r['gt']!r}")
    bad = df.get('suspected_bad_gt', [])
    print()
    print('suspected bad GT (top 10):')
    for r in bad[:10]:
        print(f"  {r['sample_id']}  score={r['score']:.3f}  gt={r['gt']!r}")
    cvl = df.get('cer_vs_length', {})
    if cvl:
        plt.figure()
        for m, series in cvl.items():
            plt.plot([p[0] for p in series], [p[1] for p in series], marker='o', label=m)
        plt.xlabel('gt length (chars)'); plt.ylabel('CER'); plt.legend()
        plt.title('CER vs length'); plt.show()
    cd = df.get('cer_distribution', {})
    bx = [{'label': f'{m}/{ds}', 'med': s['median'], 'q1': s['q1'], 'q3': s['q3'],
           'whislo': s['min'], 'whishi': s['max'], 'fliers': []}
          for m, per in cd.items() for ds, s in per.items()]
    if bx:
        fig, ax = plt.subplots(figsize=(6, 1.5 + 0.4 * len(bx)))
        ax.bxp(bx, vert=False, showfliers=False)
        ax.set_xlabel('CER'); ax.set_title('CER distribution (model x dataset)'); plt.show()

## 6. Model landscape

In [ ]:
ml = F.get('model_landscape', {})
if not ml:
    print('model landscape not available')
else:
    import numpy as np
    ag = ml.get('agreement', {})
    if ag.get('identity'):
        mm = ag.get('models', [])
        A = np.array(ag['identity'], float)
        fig, ax = plt.subplots()
        im = ax.imshow(A, vmin=0, vmax=1, cmap='viridis')
        ax.set_xticks(range(len(mm))); ax.set_xticklabels(mm, rotation=45, ha='right')
        ax.set_yticks(range(len(mm))); ax.set_yticklabels(mm)
        fig.colorbar(im, ax=ax); ax.set_title('Pairwise identical-output rate'); plt.show()
    ens = ml.get('ensemble', {})
    single = ens.get('single', {})
    if single:
        names = list(single) + ['oracle', 'rover']
        vals = [single[m] for m in single] + [ens.get('oracle', np.nan), ens.get('rover', np.nan)]
        cols = ['C0'] * len(single) + ['C1', 'C2']
        plt.figure(); plt.bar(range(len(names)), vals, color=cols)
        plt.xticks(range(len(names)), names, rotation=45, ha='right')
        plt.ylabel('CER'); plt.title('Single models vs oracle / ROVER'); plt.show()
    bc = ml.get('bootstrap_cer', {})
    rows = [(f'{m}/{ds}', d['point'], d['lo'], d['hi'])
            for m, per in bc.items() for ds, d in per.items()]
    if rows:
        y = np.arange(len(rows)); pt = np.array([r[1] for r in rows])
        lo = pt - np.array([r[2] for r in rows]); hi = np.array([r[3] for r in rows]) - pt
        plt.figure(figsize=(6, 1.5 + 0.4 * len(rows)))
        plt.errorbar(pt, y, xerr=[lo, hi], fmt='o', capsize=3)
        plt.yticks(y, [r[0] for r in rows]); plt.xlabel('CER (bootstrap 95% CI)')
        plt.title('Bootstrap CER CIs'); plt.show()
    bt = ml.get('bradley_terry', {})
    if bt:
        ks = list(bt); plt.figure(); plt.bar(range(len(ks)), [bt[k] for k in ks])
        plt.xticks(range(len(ks)), ks); plt.ylabel('BT strength')
        plt.title('Bradley-Terry ability'); plt.show()

## 7. Operations

In [ ]:
op = F.get('operational', {})
if not op:
    print('operational not available')
else:
    import numpy as np
    par = op.get('pareto', {})
    if par:
        floor = 1e-3
        xs = [max(d.get('usd_per_1k', 0.0), floor) for d in par.values()]
        ys = [d.get('cer_light', 0.0) for d in par.values()]
        plt.figure(); plt.scatter(xs, ys)
        for m, x, yv in zip(par.keys(), xs, ys):
            plt.annotate(m, (x, yv), fontsize=8, xytext=(3, 3), textcoords='offset points')
        plt.xscale('log'); plt.xlabel('USD / 1k lines (0 floored to 1e-3)')
        plt.ylabel('CER (light)'); plt.title('Cost-quality Pareto'); plt.show()
    lat = op.get('latency', {})
    if lat:
        ms = list(lat); x = np.arange(len(ms)); w = 0.35
        plt.figure()
        plt.bar(x - w / 2, [lat[m].get('p50', np.nan) for m in ms], width=w, label='p50')
        plt.bar(x + w / 2, [lat[m].get('p95', np.nan) for m in ms], width=w, label='p95')
        plt.xticks(x, ms); plt.ylabel('ms'); plt.legend(); plt.title('Latency'); plt.show()
    fc = op.get('failure_causes', {})
    classes = ['ok', 'empty', 'refusal', 'rate_limited', 'prompt_echo',
               'meta_commentary', 'wrong_script', 'other_error']
    rows = []
    for m, per in fc.items():
        agg = {c: 0 for c in classes}
        for d in per.values():
            for c in classes:
                agg[c] += d.get(c, 0)
        rows.append((m, agg))
    if rows:
        x = np.arange(len(rows)); bottom = np.zeros(len(rows))
        plt.figure()
        for c in classes:
            v = np.array([r[1][c] for r in rows], float)
            plt.bar(x, v, bottom=bottom, label=c); bottom += v
        plt.xticks(x, [r[0] for r in rows]); plt.legend(fontsize=7, ncol=2)
        plt.title('Failure causes (8 classes, summed over datasets)'); plt.show()

## 8. Corpus health

In [ ]:
ch = F.get('corpus_health', {})
if not ch:
    print('corpus health not available')
else:
    gt = ch.get('gt_stats', {})
    if gt:
        plt.figure()
        for ds, d in gt.items():
            z = d.get('zipf', [])
            if z:
                plt.plot([p[0] for p in z], [p[1] for p in z], marker='.', label=ds)
        plt.xscale('log'); plt.yscale('log')
        plt.xlabel('rank'); plt.ylabel('frequency'); plt.legend()
        plt.title('Zipf (log-log) per dataset'); plt.show()
    churn = ch.get('leaderboard_churn', {})
    metrics = churn.get('metrics', []); ranks = churn.get('ranks', {})
    if metrics and ranks:
        plt.figure()
        for m, rr in ranks.items():
            plt.plot(range(len(metrics)), [rr.get(k) for k in metrics], marker='o', label=m)
        plt.xticks(range(len(metrics)), metrics, rotation=45, ha='right')
        plt.ylabel('rank'); plt.gca().invert_yaxis(); plt.legend()
        plt.title('Leaderboard churn across metrics'); plt.show()

## How to read these — caveats

- The **Coverage** grid gates everything: a section computed over a
  half-scored (model, dataset) pair is a preview, not a verdict — read
  every later chart against how full that cell is.
- **Bootstrap CIs** use a fixed seed (1234) and percentile intervals, so
  they are reproducible but still narrow on small n; overlapping bars
  mean 'not distinguishable here', not 'equal'.
- **Failure-class / refusal detection is heuristic** — an ordered rule set
  over the raw output text. It catches the common shapes (empty, refusal,
  prompt echo, meta commentary, wrong script, rate-limited) and lumps the
  rest into `other_error`; treat the split as indicative.
- **Suspected-bad-GT** is a disagreement score (models agree with each
  other but not the reference), not a proven label — eyeball the lines.
- Contamination is never computed here; it needs an external overlap file
  and stays out of this notebook.